# Ppto27 · 30 Gold (modelo del Power BI) — PySpark
Modelo estrella con **las mismas tablas y columnas** que los CSV de ejemplo (el Power BI sólo cambia el origen):
`ppto27_gld_fact_ppto` (contable) · `ppto27_gld_fact_gestion_dirind` · `ppto27_gld_fact_flags` · `ppto27_gld_dim_*`.
* Moneda de origen, moneda local y PEN con el tipo de cambio maestro (México en PEN). Nivel 1 / Nivel 2 en Formato Perú.
* Gestión Dir/Ind con los drivers v2: imputación directa, soporte directo e indirecto repartido por país y mes.

In [ ]:
%pip install openpyxl --quiet

In [ ]:
import sys
sys.path.append('/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27/00_codigo')            # librerías ppto27_*.py subidas al Volume
import time
import ppto27_lib as L, ppto27_spark as SP

t = time.time()
for n, filas in SP.gold(spark).items():
    print(f'ppto27_gld_{n}: {filas:,} filas')
print(f'{time.time() - t:.0f} s')

In [ ]:
display(spark.sql(f"""
SELECT p.orden_n1, p.n1_formato, p.orden_n2, p.n2_formato, f.id_pais, ROUND(SUM(f.monto_pen), 0) AS ppto_2027_pen
FROM {L.tabla('gld_fact_ppto')} f JOIN {L.tabla('gld_dim_linea_pyg')} p USING (id_linea_pyg)
WHERE f.estado_linea IN ('OK', 'OK con alerta') GROUP BY ALL ORDER BY p.orden_n1, p.orden_n2, f.id_pais"""))